# Cognitive Reinforcement Learning: Bandits and the Prisoner's Dilemma

A modular planner/executor/validator agent (arxiv:2310.00194 — see the
"Modular Planner/Executor/Validator Agent" tutorial) reused as-is here, paired
with a **Q-learning environment** implemented as a `psychscanner.feedback.FeedbackBase`
handler: each round's reward is computed from the agent's choice, folded into
a running value estimate via the standard update
`Q[a] += lr * (reward - Q[a])`, and reported back as feedback text the agent
reads before its next choice — the same feedback-injection mechanism the RM
tutorials use, just carrying numeric reward instead of correctness feedback.

Two classic cognitive-science reward-learning paradigms:
1. **n-armed bandit** — the explore/exploit tradeoff central to human
   reinforcement-learning research (e.g. Schulz & Gershman's work on
   directed vs. random exploration).
2. **Iterated Prisoner's Dilemma vs. Tit-for-Tat** — repeated economic games
   are how recent work (Akata, Schulz et al., *"Playing Repeated Games with
   Large Language Models"*, 2023) evaluates whether LLMs reproduce classic
   game-theoretic strategies under reward feedback.

No RL library is used: the "environment" is a fixed reward/payoff table and
the update is two lines of arithmetic — reach for `gymnasium`/`stable-baselines`
only once the environment grows more complex than that.

In [1]:
import random
import re
from pathlib import Path

import psychscanner as psy
from psychscanner.agents import make_planner_executor_agent
from psychscanner.feedback import FeedbackBase
from psychscanner.memories import llm_chat_model

RUN_DIR = Path.cwd() / "_cognitive_rl_tutorial_run"
model = llm_chat_model(model="llama3.2:3b", family="ollama", parameters={"temperature": 0.7})
agent = make_planner_executor_agent(model, max_iterations=1)  # plan -> execute -> validate, no retry loop
print("PsychScanner successfully imported!")

--<api key>-- warning: OLLAMA_API_KEY not set; proceeding without explicit api_key for family 'ollama'


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model='llama3.2:3b' temperature=0.7


PsychScanner successfully imported!


## 1. Cognitive reward-learning: a 3-armed bandit

Arms `A`, `B`, `C` have hidden mean rewards (`2`, `6`, `4` + noise) the agent
must discover by trial and error. `BanditFeedback` is the "environment": it
scores the agent's choice, updates a running per-arm value estimate, and
reports both the reward and the current estimates back to the agent — so the
agent's *next* planning pass has something to reason about.

In [2]:
class BanditFeedback(FeedbackBase):
    ARM_MEANS = {"A": 2.0, "B": 6.0, "C": 4.0}

    def __init__(self, learning_rate=0.3, seed=0):
        self.lr = learning_rate
        self.rng = random.Random(seed)
        self.q = {arm: 0.0 for arm in self.ARM_MEANS}
        self.history = []  # (arm, reward) per round

    def _parse_arm(self, text: str) -> str:
        matches = re.findall(r"\b([ABC])\b", text.upper())
        return matches[-1] if matches else self.rng.choice(list(self.ARM_MEANS))

    def on_response(self, trial, response):
        arm = self._parse_arm(response["content"])
        reward = self.rng.gauss(self.ARM_MEANS[arm], 1.0)
        self.q[arm] += self.lr * (reward - self.q[arm])
        self.history.append((arm, reward))
        estimates = ", ".join(f"{a}={v:.2f}" for a, v in self.q.items())
        return (f"You chose {arm} and received reward {reward:.2f}. "
                f"Current value estimates — {estimates}.")

In [3]:
N_BANDIT_ROUNDS = 6

bandit_task = {
    "tasktype": "bandit", "taskname": "cognitive_rl_bandit",
    "instructions": {"definition": [
        "Each round, choose one arm: A, B, or C. State your final choice clearly as a single letter."
    ]},
    "contexts": ["3-armed bandit"], "contexts_id": ["bandit"], "context_present": False,
    "chain_type": "task", "parser": "0",
    "items": {"bandit": [
        {"trcode": f"bandit_{i}", "stimulus": f"Round {i}: choose A, B, or C.", "fb": True}
        for i in range(1, N_BANDIT_ROUNDS + 1)
    ]},
}

bandit_card = psy.ExpCardInit()
bandit_card.proj_dir, bandit_card.projectname = RUN_DIR, "bandit_demo"
bandit_card.task_file = bandit_task
bandit_card.cogtype, bandit_card.nsim = "no", 1
bandit_card.chain_type, bandit_card.memory = "task", "Convo"
bandit_card.feedback, bandit_card.feedback_fn = True, BanditFeedback

bandit_scanner = psy.ScannerModel(expcard=psy.ExpCard(bandit_card))
bandit_results = bandit_scanner.run(custom_agent=agent)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_cognitive_rl_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_cognitive_rl_tutorial_run/bandit_demo/cognitive_rl_bandit/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

6it [08:03, 74.87s/it]

6it [08:03, 80.65s/it]


2026-07-06 16:43:50.877 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 16:43:50.902 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


In [4]:
fb_handler = None
for trial in bandit_results[0]:
    print(trial["trcode"], "->", trial["fb_response"])

print("\nTrue means:", BanditFeedback.ARM_MEANS)

bandit_1 -> You chose B and received reward 6.94. Current value estimates — A=0.00, B=2.08, C=0.00.
bandit_2 -> You chose B and received reward 4.60. Current value estimates — A=0.00, B=2.84, C=0.00.
bandit_3 -> You chose B and received reward 5.32. Current value estimates — A=0.00, B=3.58, C=0.00.
bandit_4 -> You chose B and received reward 6.37. Current value estimates — A=0.00, B=4.42, C=0.00.
bandit_5 -> You chose A and received reward 0.98. Current value estimates — A=0.30, B=4.42, C=0.00.
bandit_6 -> You chose B and received reward 5.93. Current value estimates — A=0.30, B=4.87, C=0.00.

True means: {'A': 2.0, 'B': 6.0, 'C': 4.0}


With only a handful of rounds and no explicit exploration bonus in the
prompt, don't expect textbook uniform exploration — a small local model
often locks onto the first arm that pays off well rather than sampling every
arm, which is itself a real (if unflattering) finding about directed vs.
random exploration in LLM-driven decision-making, the kind of comparison
Schulz's exploration-strategy work makes with human subjects.

## 2. Iterated Prisoner's Dilemma vs. Tit-for-Tat

Classic payoffs: mutual cooperation `3/3`, mutual defection `1/1`, one-sided
defection `5/0`. The opponent plays Tit-for-Tat — it mirrors the agent's
*previous* move (cooperating on round 1) — the same reference strategy
Akata/Schulz compare LLM behavior against.

In [5]:
class PDFeedback(FeedbackBase):
    PAYOFFS = {  # (my_move, opponent_move) -> my_payoff
        ("cooperate", "cooperate"): 3, ("cooperate", "defect"): 0,
        ("defect", "cooperate"): 5, ("defect", "defect"): 1,
    }

    def __init__(self):
        self.my_prev_move = None
        self.total_payoff = 0
        self.history = []  # (my_move, opponent_move, payoff)

    def _parse_move(self, text: str) -> str:
        matches = re.findall(r"\b(cooperate|defect)\b", text.lower())
        return matches[-1] if matches else "cooperate"

    def on_response(self, trial, response):
        my_move = self._parse_move(response["content"])
        opponent_move = self.my_prev_move or "cooperate"  # Tit-for-Tat: mirrors my last move
        payoff = self.PAYOFFS[(my_move, opponent_move)]
        self.total_payoff += payoff
        self.history.append((my_move, opponent_move, payoff))
        self.my_prev_move = my_move
        return (f"You played {my_move}; the opponent (Tit-for-Tat) played {opponent_move}. "
                f"Your payoff this round: {payoff}. Running total: {self.total_payoff}.")

In [6]:
N_PD_ROUNDS = 6

pd_task = {
    "tasktype": "game", "taskname": "cognitive_rl_pd",
    "instructions": {"definition": [
        "You are playing a repeated game against another player. Each round, choose to cooperate or defect. "
        "State your final choice clearly as one word: COOPERATE or DEFECT."
    ]},
    "contexts": ["Iterated Prisoner's Dilemma"], "contexts_id": ["pd"], "context_present": False,
    "chain_type": "task", "parser": "0",
    "items": {"pd": [
        {"trcode": f"pd_{i}", "stimulus": f"Round {i} of the repeated game.", "fb": True}
        for i in range(1, N_PD_ROUNDS + 1)
    ]},
}

pd_card = psy.ExpCardInit()
pd_card.proj_dir, pd_card.projectname = RUN_DIR, "pd_demo"
pd_card.task_file = pd_task
pd_card.cogtype, pd_card.nsim = "no", 1
pd_card.chain_type, pd_card.memory = "task", "Convo"
pd_card.feedback, pd_card.feedback_fn = True, PDFeedback

pd_scanner = psy.ScannerModel(expcard=psy.ExpCard(pd_card))
pd_results = pd_scanner.run(custom_agent=agent)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_cognitive_rl_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_cognitive_rl_tutorial_run/pd_demo/cognitive_rl_pd/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

6it [08:22, 91.34s/it]

6it [08:22, 83.82s/it]


2026-07-06 16:52:14.147 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 16:52:14.233 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


In [7]:
for trial in pd_results[0]:
    print(trial["trcode"], "->", trial["fb_response"])

pd_1 -> You played cooperate; the opponent (Tit-for-Tat) played cooperate. Your payoff this round: 3. Running total: 3.
pd_2 -> You played cooperate; the opponent (Tit-for-Tat) played cooperate. Your payoff this round: 3. Running total: 6.
pd_3 -> You played cooperate; the opponent (Tit-for-Tat) played cooperate. Your payoff this round: 3. Running total: 9.
pd_4 -> You played defect; the opponent (Tit-for-Tat) played cooperate. Your payoff this round: 5. Running total: 14.
pd_5 -> You played defect; the opponent (Tit-for-Tat) played defect. Your payoff this round: 1. Running total: 15.
pd_6 -> You played cooperate; the opponent (Tit-for-Tat) played defect. Your payoff this round: 0. Running total: 15.


## Recap

- No new agent architecture needed here — `make_planner_executor_agent`
  (arxiv:2310.00194) is reused unchanged; the RL machinery lives entirely in
  a `FeedbackBase` subclass, which is exactly what that class already exists
  for (see the RM-with-feedback tutorial).
- The bandit's `Q[a] += lr * (reward - Q[a])` and the iterated-game payoff
  lookup are the whole "RL library" this needs — reach for `gymnasium` or
  `stable-baselines` only if the environment stops being a fixed table.
- Both paradigms plug into `ScannerModel.run(custom_agent=...)` exactly like
  any other feedback-driven task — `feedback=True` + `feedback_fn=<class>`
  is the only thing that changed from a plain survey run.